# Medical Adherence Predictor
## End-to-End Machine Learning Pipeline

**Dataset:** [Mendeley Data – Medication Adherence](https://data.mendeley.com/datasets/zkp7sbbx64/2)  
**Goal:** Predict whether a patient will adhere to their medication regimen

---

### Table of Contents
1. [Setup & Imports](#setup)
2. [Data Loading & Understanding](#data)
3. [Data Preprocessing](#preprocessing)
4. [Feature Engineering](#features)
5. [Exploratory Data Analysis](#eda)
6. [Model Training](#training)
7. [Model Evaluation](#evaluation)
8. [Feature Importance](#importance)
9. [Export for Power BI](#export)

## 1. Setup & Imports <a id='setup'></a>

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, roc_curve)

# Notebook-friendly plot settings
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})
sns.set_style('whitegrid')

print('[OK] All imports successful!')

## 2. Data Loading & Understanding <a id='data'></a>

**Place your downloaded dataset at:** `../data/raw/medication_adherence.csv`

If not found, the pipeline auto-generates a synthetic dataset mirroring the real data's structure.

In [ ]:
# Run preprocessing to load/generate data
import subprocess
subprocess.run(['python', '../src/preprocessing.py'], capture_output=False)

# Load the raw dataset for exploration
df = pd.read_csv('../data/raw/medication_adherence.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Data types and missing values
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'null_count': df.isnull().sum(),
    'null_%': (df.isnull().sum() / len(df) * 100).round(2),
    'unique': df.nunique()
})
print('Data Overview:')
info_df

In [ ]:
# Target variable
print('Target Variable: adherent')
print(df['adherent'].value_counts())
print(f'\nAdherence rate: {df["adherent"].mean():.1%}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df['adherent'].value_counts().plot.bar(ax=axes[0], color=['#E74C3C', '#27AE60'])
axes[0].set_title('Adherence Count')
axes[0].set_xticklabels(['Non-Adherent (0)', 'Adherent (1)'], rotation=0)

df['adherent'].value_counts().plot.pie(ax=axes[1],
    labels=['Non-Adherent', 'Adherent'],
    colors=['#E74C3C', '#27AE60'], autopct='%1.1f%%')
axes[1].set_title('Adherence Distribution')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing <a id='preprocessing'></a>

In [ ]:
# Load cleaned data
cleaned = pd.read_csv('../data/processed/cleaned_data.csv')
print(f'Cleaned data shape: {cleaned.shape}')
cleaned.head()

## 4. Feature Engineering <a id='features'></a>

In [ ]:
subprocess.run(['python', '../src/feature_engineering.py'], capture_output=False)

featured = pd.read_csv('../data/processed/featured_data.csv')
print(f'Featured data shape: {featured.shape}')

# Show new features
new_cols = ['refill_ratio', 'financial_burden', 'age_group',
            'medication_complexity', 'supply_category', 'refill_gap']
featured[new_cols + ['adherent']].describe().round(3)

## 5. Exploratory Data Analysis <a id='eda'></a>

In [ ]:
# Correlation heatmap
num_cols = ['adherent', 'refill_ratio', 'financial_burden', 'age_group',
            'refill_gap', 'medication_complexity', 'supply_category']
num_cols = [c for c in num_cols if c in featured.columns]

fig, ax = plt.subplots(figsize=(9, 7))
corr = featured[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Refill ratio by adherence
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label, color in [(0, '#E74C3C'), (1, '#27AE60')]:
    subset = featured[featured['adherent'] == label]['refill_ratio']
    axes[0].hist(subset, bins=30, alpha=0.6, color=color,
                 label=f'{"Adherent" if label else "Non-Adherent"}')
axes[0].set_title('Refill Ratio by Adherence', fontweight='bold')
axes[0].set_xlabel('Refill Ratio')
axes[0].legend()

featured.boxplot(column='financial_burden', by='adherent', ax=axes[1])
axes[1].set_title('Financial Burden by Adherence', fontweight='bold')
axes[1].set_xlabel('Adherent (0=No, 1=Yes)')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 6. Model Training <a id='training'></a>

In [ ]:
subprocess.run(['python', '../src/train.py'], capture_output=False)
print('\n[OK] Training complete!')

## 7. Model Evaluation <a id='evaluation'></a>

In [ ]:
subprocess.run(['python', '../src/evaluate.py'], capture_output=False)

# Load metrics
metrics = pd.read_csv('../outputs/reports/model_metrics.csv')
print('\nModel Performance Summary:')
metrics

In [ ]:
# Display saved plots
from IPython.display import Image, display
for img_path in ['../outputs/figures/07_roc_curves.png',
                  '../outputs/figures/09_feature_importance.png']:
    display(Image(img_path, width=700))

## 8. Feature Importance <a id='importance'></a>

### Key Findings:

| Feature | Importance | Insight |
|---------|-----------|--------|
| `refill_ratio` | High | Most direct behavioral signal of adherence |
| `financial_burden` | High | High costs → patients skip refills |
| `annual_contribution` | Medium | Insurance cost affects adherence |
| `age_group` | Medium | Elderly patients face more barriers |
| `refill_gap` | Medium | Absolute gap shows severity of non-adherence |

## 9. Export for Power BI <a id='export'></a>

In [ ]:
predictions = pd.read_csv('../outputs/reports/final_predictions.csv')
print(f'Power BI export: {predictions.shape}')
print(f'Columns: {list(predictions.columns)}')
predictions.head()

---
## Summary

This notebook demonstrated a complete ML pipeline:
1. **Data Loading** → Raw CSV with patient adherence data
2. **Preprocessing** → Imputation, encoding, scaling, outlier handling
3. **Feature Engineering** → refill_ratio, financial_burden, age_group, etc.
4. **EDA** → Distribution plots, correlation analysis
5. **Modeling** → Logistic Regression, Decision Tree, Random Forest
6. **Evaluation** → Accuracy, Precision, Recall, F1, ROC-AUC
7. **Interpretation** → Feature importance from Random Forest
8. **Export** → CSV for Power BI dashboard

**Key Insight:** Recall is the most critical metric in healthcare — minimizing False Negatives (missed non-adherent patients) is paramount.